# 유사 사례 비교 분석 뉴스 수집

이 노트북은 보고서의 "유사 사례 비교 분석" 섹션에서 키워드를 추출하고, 딥서치뉴스 API를 통해 관련 뉴스를 검색하여 데이터베이스의 `news` 컬럼에 저장합니다.

In [34]:
# 필요한 라이브러리 import
import pandas as pd
import requests
import json
import re
from sqlalchemy import create_engine, text, inspect
from datetime import datetime, timedelta
import time
from tqdm import tqdm

# 형태소 분석기 설정
# ⚠️ CRITICAL: KoNLPy import 자체가 JPype1을 초기화하려고 시도하여 커널이 죽습니다
# Python의 try-except로는 막을 수 없습니다 (import 시점에 JVM이 시작됨)
# 따라서 KoNLPy를 전혀 import하지 않고 정규표현식만 사용합니다
# 정규표현식으로도 법률 문서의 핵심 키워드를 효과적으로 추출할 수 있습니다
MORPH_AVAILABLE = False
morph_analyzer = None
MORPH_TYPE = "정규표현식"

print("ℹ️ 정규표현식 기반 키워드 추출을 사용합니다.")
print("  (KoNLPy는 JPype1 문제로 인해 사용하지 않습니다)")

# 딥서치뉴스 API 설정 (환경 변수 사용)
import os
from dotenv import load_dotenv
load_dotenv()  # report/.env
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), "service", ".env"))  # service/.env 폴백 (DB_URL 등)
DEEPSEARCH_API_KEY = os.getenv("DEEPSEARCH_API_KEY")
if not DEEPSEARCH_API_KEY:
    raise ValueError("환경변수 DEEPSEARCH_API_KEY가 없습니다. report/.env에 설정하세요.")
DEEPSEARCH_API_BASE_URL = "https://api-v2.deepsearch.com"
DEEPSEARCH_API_ENDPOINT = "/v1/articles"

# 데이터베이스 연결 (환경 변수 사용, report/.env 또는 service/.env)
DB_URL = os.getenv("DB_URL")
if not DB_URL:
    raise ValueError("환경변수 DB_URL이 없습니다. report/.env 또는 service/.env에 설정하세요.")
engine = create_engine(DB_URL)

print("라이브러리 import 완료")

ℹ️ 정규표현식 기반 키워드 추출을 사용합니다.
  (KoNLPy는 JPype1 문제로 인해 사용하지 않습니다)
라이브러리 import 완료


In [35]:
# news 컬럼이 없으면 추가
inspector = inspect(engine)
columns = [col['name'] for col in inspector.get_columns('final_training_data_copy_sample10_md')]

if 'news' not in columns:
    print("news 컬럼이 없습니다. 추가 중...")
    with engine.connect() as conn:
        conn.execute(text("ALTER TABLE public.final_training_data_copy_sample10_md ADD COLUMN IF NOT EXISTS news TEXT"))
        conn.commit()
    print("news 컬럼 추가 완료")
else:
    print("news 컬럼이 이미 존재합니다.")

news 컬럼이 이미 존재합니다.


/tmp/ipykernel_2840935/3749243459.py:3: SAWarning: Did not recognize type 'vector' of column 'summary_embedding'
  columns = [col['name'] for col in inspector.get_columns('final_training_data_copy_sample10_md')]


In [36]:
def extract_similar_cases_section(report_md: str) -> str:
    """
    report_md에서 유사 사례 비교 분석 섹션(섹션 4)을 추출합니다.
    """
    if not report_md or report_md.strip() == "":
        return ""
    
    # 섹션 4 추출: ## 4. 유사 사례 비교분석
    pattern = r'##\s+4\.\s+유사\s+사례\s+비교분석(.*?)(?=##\s+5\.|$)'
    match = re.search(pattern, report_md, re.DOTALL | re.IGNORECASE)
    
    if match:
        return match.group(1).strip()
    return ""

def extract_nouns_with_morph(text: str) -> list:
    """
    형태소 분석기를 사용하여 텍스트에서 명사만 추출합니다.
    형태소 분석기가 없으면 정규표현식으로 대체합니다.
    """
    if not text or not text.strip():
        return []
    
    nouns = []
    
    if MORPH_AVAILABLE and morph_analyzer:
        try:
            # 형태소 분석으로 명사 추출
            # Komoran과 Okt의 API가 약간 다르므로 통합 처리
            if MORPH_TYPE == "Komoran":
                morphs = morph_analyzer.pos(text)
            else:  # Okt
                morphs = morph_analyzer.pos(text, norm=True, stem=True)
            # 명사(Noun)만 추출
            nouns = [word for word, pos in morphs if pos.startswith('N')]
        except Exception as e:
            print(f"⚠️ 형태소 분석 오류: {e}, 정규표현식으로 대체합니다.")
            # 폴백: 정규표현식 사용
            nouns = re.findall(r'[\w가-힣]{2,}', text)
    else:
        # 형태소 분석기가 없으면 정규표현식 사용
        # 조사/어미가 붙은 단어에서 조사/어미 제거
        words = re.findall(r'[\w가-힣]{2,}', text)
        
        # 조사/어미 패턴 제거
        # 예: '등기에' -> '등기', '보호에' -> '보호'
        josa_patterns = [
            r'에$', r'을$', r'를$', r'의$', r'이$', r'가$', r'은$', r'는$',
            r'와$', r'과$', r'으로$', r'로$', r'에서$', r'부터$', r'까지$',
            r'에게$', r'께$', r'한테$', r'더러$', r'에게서$', r'한테서$',
            r'도$', r'만$', r'조차$', r'마저$', r'까지$', r'부터$',
            r'처럼$', r'같이$', r'만큼$', r'커녕$', r'따라$', r'따름$'
        ]
        
        cleaned_words = []
        for word in words:
            # 조사/어미 제거
            cleaned = word
            for pattern in josa_patterns:
                cleaned = re.sub(pattern, '', cleaned)
            
            # 조사 제거 후에도 의미 있는 단어인지 확인
            if len(cleaned) >= 2 and cleaned != word:
                cleaned_words.append(cleaned)
            elif len(word) >= 2:
                # 조사가 없는 경우 원본 단어 사용
                cleaned_words.append(word)
        
        nouns = cleaned_words
    
    return nouns

def extract_keywords_from_bill_data(bill_row: dict, similar_cases_section: str = "") -> list:
    """
    형태소 분석기를 사용하여 법률안의 핵심 키워드(명사)만 추출합니다.
    법안명의 핵심 키워드를 우선적으로 사용하여 더 정확한 뉴스 검색을 합니다.
    """
    keywords = []
    
    # 법률 문서에서 자주 나오지만 검색에 도움이 안 되는 stopwords
    stopwords = {
        # 조사/접속사
        '등', '및', '에', '의', '을', '를', '이', '가', '은', '는', '와', '과',
        '으로', '로', '에서', '부터', '까지', '때문', '위해', '대한', '관한', '위한',
        '등에', '등을', '등의', '등으로', '등에서', '에의', '에의한',
        # 법률 문서 특수 용어
        '법률', '법안', '개정', '안', '법', '일부개정법률안', '전부개정법률안',
        '일부개정을', '주요내용', '제안이유', '따르면', '위하여',
        # 일반적인 문서 용어
        '목표', '정부', '일반', '구조', '환경', '이후', '이전', '이상', '이하',
        '각각', '함께', '반면', '가장', '최근', '들어서', '폭증하고',
        '이유로', '사용자가', '근로자임을', '보유에', '대안의',
        '구성된', '종사하는', '기여한', '제출하고', '심의를', '허가를',
        # 의미 없는 단어
        '문장은', '정비를', '용어', '과제', '금지', '읽고', '습관',
        '구조적', '환경적', '연도별', '5년간', '3년간', '1년간',
        # 숫자/날짜
        '2005년', '16',
        # 동사 형태
        '읽고', '보고', '하고', '되어', '되는', '하는',
        # 조사가 붙은 형태
        '등기에', '등기의', '등기를', '등기로', '등기에서',
        '보호에', '보호의', '보호를', '보호로',
        '법에', '법의', '법을', '법으로'
    }
    
    # 1. 법안명에서 핵심 키워드만 추출 (가장 중요)
    if bill_row.get('bill_name'):
        bill_name = str(bill_row['bill_name']).strip()
        # 괄호와 괄호 안 내용 제거
        bill_name_clean = re.sub(r'\([^)]*\)', '', bill_name).strip()
        # "일부개정법률안", "전부개정법률안", "법률안", "법률" 등 완전히 제거
        bill_name_clean = re.sub(r'\s*일부개정법률안\s*', ' ', bill_name_clean)
        bill_name_clean = re.sub(r'\s*전부개정법률안\s*', ' ', bill_name_clean)
        bill_name_clean = re.sub(r'\s*법률안\s*', ' ', bill_name_clean)
        bill_name_clean = re.sub(r'\s*법률\s*', ' ', bill_name_clean)
        bill_name_clean = re.sub(r'\s+', ' ', bill_name_clean).strip()
        
        if bill_name_clean and len(bill_name_clean) > 2:
            # 형태소 분석기로 명사 추출
            nouns = extract_nouns_with_morph(bill_name_clean)
            
            # 필터링: 길이, stopwords, 의미 있는 키워드만 선택
            filtered_nouns = []
            for noun in nouns:
                noun_clean = noun.strip()
                # 길이 체크 (2-20자)
                if not (2 <= len(noun_clean) <= 20):
                    continue
                # stopwords 체크
                if noun_clean in stopwords:
                    continue
                # 숫자만 있는 경우 제외
                if noun_clean.isdigit():
                    continue
                # "일부개정법률안" 등 포함된 키워드 제거
                if '일부개정법률안' in noun_clean or '전부개정법률안' in noun_clean or '법률안' in noun_clean:
                    continue
                filtered_nouns.append(noun_clean)
            
            # 법안명 키워드를 우선적으로 추가 (최대 5개)
            keywords.extend(filtered_nouns[:5])
    
    # 2. 발의자명은 제외 (연예인 이름 등과 혼동될 수 있음)
    # 발의자명은 뉴스 검색에 사용하지 않음
    
    # 3. summary는 제외 (너무 일반적인 키워드가 많이 포함됨)
    # summary에서 키워드를 추출하지 않음
    
    # 2. 발의자명은 제외 (연예인 이름 등과 혼동될 수 있음)
    # 발의자명은 뉴스 검색에 사용하지 않음
    
    # 3. summary는 제외 (너무 일반적인 키워드가 많이 포함됨)
    # summary에서 키워드를 추출하지 않음
    
    # 4. 유사 사례 비교 분석 섹션에서 법안명 추출 (보조적으로만 사용)
    if similar_cases_section:
        # 테이블에서 법안명 추출
        table_pattern = r'\|\s*\d+\s*\|\s*([^\|]+)\s*\|\s*([^\|]+)\s*\|\s*[\d.]+\s*\|\s*([^\|]+)\s*\|'
        matches = re.finditer(table_pattern, similar_cases_section)
        
        similar_case_keywords = []
        for match in matches:
            bill_name = match.group(1).strip()
            # 법안명에서 핵심 키워드 추출
            bill_name_clean = re.sub(r'\([^)]*\)', '', bill_name).strip()
            # "일부개정법률안", "전부개정법률안", "법률안", "법률" 등 완전히 제거
            bill_name_clean = re.sub(r'\s*일부개정법률안\s*', ' ', bill_name_clean)
            bill_name_clean = re.sub(r'\s*전부개정법률안\s*', ' ', bill_name_clean)
            bill_name_clean = re.sub(r'\s*법률안\s*', ' ', bill_name_clean)
            bill_name_clean = re.sub(r'\s*법률\s*', ' ', bill_name_clean)
            bill_name_clean = re.sub(r'\s+', ' ', bill_name_clean).strip()
            
            if bill_name_clean and len(bill_name_clean) > 2:
                # 형태소 분석기로 명사 추출
                nouns = extract_nouns_with_morph(bill_name_clean)
                
                # 필터링: 길이, stopwords 체크
                filtered_nouns = []
                for noun in nouns:
                    noun_clean = noun.strip()
                    # 길이 체크 (2-20자)
                    if not (2 <= len(noun_clean) <= 20):
                        continue
                    # stopwords 체크
                    if noun_clean in stopwords:
                        continue
                    # 숫자만 있는 경우 제외
                    if noun_clean.isdigit():
                        continue
                    # "일부개정법률안" 등 포함된 키워드 제거
                    if '일부개정법률안' in noun_clean or '전부개정법률안' in noun_clean or '법률안' in noun_clean:
                        continue
                    filtered_nouns.append(noun_clean)
                
                similar_case_keywords.extend(filtered_nouns)
        
        # 유사 사례 키워드는 현재 법안명 키워드와 공통된 키워드만 추가
        # (완전히 다른 법안의 키워드는 제외하여 검색 정확도 향상)
        existing_keywords_lower = [k.lower() for k in keywords]
        
        # 현재 법안명의 핵심 키워드 추출 (비교용)
        current_bill_keywords = []
        if bill_row.get('bill_name'):
            bill_name = str(bill_row['bill_name']).strip()
            bill_name_clean = re.sub(r'\([^)]*\)', '', bill_name).strip()
            bill_name_clean = re.sub(r'\s*일부개정법률안\s*', ' ', bill_name_clean)
            bill_name_clean = re.sub(r'\s*전부개정법률안\s*', ' ', bill_name_clean)
            bill_name_clean = re.sub(r'\s*법률안\s*', ' ', bill_name_clean)
            bill_name_clean = re.sub(r'\s*법률\s*', ' ', bill_name_clean)
            bill_name_clean = re.sub(r'\s+', ' ', bill_name_clean).strip()
            if bill_name_clean:
                current_bill_keywords = extract_nouns_with_morph(bill_name_clean)
                current_bill_keywords = [k.strip() for k in current_bill_keywords if len(k.strip()) >= 2]
        
        # 유사 사례 키워드 중 현재 법안명과 공통 키워드가 있거나, 
        # 법안명 키워드가 부족한 경우에만 추가 (최대 3개)
        added_count = 0
        for kw in similar_case_keywords:
            if kw.lower() in existing_keywords_lower:
                continue
            
            # 현재 법안명 키워드와 공통된 부분이 있는지 확인
            kw_lower = kw.lower()
            has_common = any(
                kw_lower in ck.lower() or ck.lower() in kw_lower 
                for ck in current_bill_keywords 
                if len(ck) >= 2
            )
            
            # 공통 키워드가 있거나, 법안명 키워드가 3개 미만인 경우에만 추가
            if has_common or len(keywords) < 3:
                if len(keywords) < 10:  # 전체 키워드 수 제한
                    keywords.append(kw)
                    added_count += 1
                    if added_count >= 3:  # 유사 사례에서 최대 3개만 추가
                        break
    
    # 중복 제거 및 빈 문자열 제거
    keywords = list(set([k.strip() for k in keywords if k and len(k.strip()) > 1]))
    
    # 최종 필터링: 조사/어미 제거 및 품질 검증
    filtered_keywords = []
    josa_patterns = [
        r'에$', r'을$', r'를$', r'의$', r'이$', r'가$', r'은$', r'는$',
        r'와$', r'과$', r'으로$', r'로$', r'에서$', r'부터$', r'까지$',
        r'에게$', r'께$', r'한테$', r'더러$', r'에게서$', r'한테서$',
        r'도$', r'만$', r'조차$', r'마저$', r'처럼$', r'같이$', r'만큼$'
    ]
    
    for k in keywords:
        k_clean = k.strip()
        
        # 길이 체크 (2-20자)
        if not (2 <= len(k_clean) <= 20):
            continue
        
        # 조사/어미 제거 (정규표현식으로 추출된 경우 대비)
        original = k_clean
        for pattern in josa_patterns:
            k_clean = re.sub(pattern, '', k_clean)
        
        # 조사 제거 후 길이 재확인
        if len(k_clean) < 2:
            continue
        
        # stopwords 체크 (조사 제거 전후 모두 확인)
        if k_clean in stopwords or original in stopwords:
            continue
        
        # "일부개정법률안", "전부개정법률안" 포함된 키워드 완전히 제거
        if '일부개정법률안' in k_clean or '전부개정법률안' in k_clean or '법률안' in k_clean:
            continue
        
        # 숫자만 있는 키워드 제거
        if k_clean.isdigit():
            continue
        
        # 조사로 끝나는 키워드 제거 (추가 안전장치)
        if any(k_clean.endswith(josa.replace('$', '').replace('\\', '')) for josa in josa_patterns):
            continue
        
        # 긴 문장 패턴 제외 (형태소 분석으로 분리되지 않은 경우)
        if len(k_clean) > 15 and (' ' in k_clean or k_clean.count(' ') > 2):
            continue
        
        filtered_keywords.append(k_clean)
    
    return filtered_keywords[:15]  # 최대 15개 키워드 사용

# 테스트
test_bill_data = {
    'bill_name': '기간제 및 단시간근로자 보호 등에 관한 법률 일부개정법률안(김해영의원 등 11인)',
    'proposer_name': '김해영',
    'summary': '기간제 근로자와 단시간 근로자의 고용안정과 근로조건 개선을 위한 법률 개정안입니다.'
}
test_similar_cases = """
| 순위 | 법안명 | 처리 결과 | 유사도 | 핵심 차이 |
| --- | --- | --- | --- | --- |
| 1 | 헌법재판소법 일부개정법률안(정부) | 대안반영폐기 | 0.9210 | 현재 상태: 공포→대안반영폐기 |
| 2 | 부동산 실권리자명의 등기에 관한 법률 일부개정법률안(정부) | 원안가결 | 0.9080 | - |
"""
keywords = extract_keywords_from_bill_data(test_bill_data, test_similar_cases)

# 키워드 추출 결과 상세 보고
print("=" * 60)
print("📊 키워드 추출 결과 보고")
print("=" * 60)
print(f"\n🔧 사용 방법: {MORPH_TYPE if MORPH_AVAILABLE else '정규표현식 (KoNLPy 미설치)'}")
print(f"📝 입력 법안명: {test_bill_data['bill_name']}")
print(f"📈 추출된 키워드 개수: {len(keywords)}개")
print(f"\n✅ 추출된 키워드 목록:")
if keywords:
    for i, kw in enumerate(keywords, 1):
        print(f"  {i:2d}. {kw}")
else:
    print("  (키워드가 추출되지 않았습니다)")
print("\n" + "=" * 60)

📊 키워드 추출 결과 보고

🔧 사용 방법: 정규표현식 (KoNLPy 미설치)
📝 입력 법안명: 기간제 및 단시간근로자 보호 등에 관한 법률 일부개정법률안(김해영의원 등 11인)
📈 추출된 키워드 개수: 3개

✅ 추출된 키워드 목록:
   1. 보호
   2. 기간제
   3. 단시간근로자



In [37]:
def filter_relevant_news(news_list: list, keywords: list, strict_mode: bool = True) -> list:
    """
    뉴스 리스트에서 법안과 관련된 뉴스만 필터링합니다.
    연예/스포츠 뉴스는 강력하게 제외합니다.
    
    Args:
        news_list: 뉴스 리스트
        keywords: 검색 키워드 리스트
        strict_mode: True면 제목에 키워드가 반드시 있어야 함, False면 요약에도 있으면 포함
    
    Returns:
        필터링된 뉴스 리스트
    """
    if not keywords or not news_list:
        return []
    
    # 키워드를 소문자로 변환 (대소문자 무시 검색)
    keywords_lower = [str(kw).lower() for kw in keywords if kw and len(str(kw)) >= 3]
    
    if not keywords_lower:
        return []
    
    # 연예/스포츠 관련 키워드 (대폭 확장)
    entertainment_terms = [
        # 연예 기본
        '연예', '가수', '배우', '아이돌', '드라마', '영화', '예능', '엔터테인먼트',
        '걸그룹', '보이그룹', '연예계', '연예인', '연예부', '연예면',
        'k-pop', 'kpop', '케이팝', '아이돌그룹',
        # 스포츠
        '스포츠', '야구', '축구', '농구', '배구', '골프', '테니스', '볼링',
        '프로야구', '프로축구', 'k리그', 'kbo', '올림픽', '월드컵', '경기', '선수', '감독',
        '야구장', '축구장', '경기장', '홈런', '골', '득점', '승리', '패배',
        # 연예인 이름 패턴 (일반적인 성씨 + 이름 패턴은 제외하되, 연예 관련 맥락에서만)
        # 드라마/영화 제목
        '드라마', '시리즈', '시즌', '에피소드', '리메이크', '원작',
        # 음악
        '앨범', '싱글', '음원', '차트', '뮤직비디오', 'mv', '콘서트', '공연',
        # 연예 뉴스 특성
        '데뷔', '컴백', '활동', '소속사', '기획사', '팬클럽', '팬미팅',
        '화제', '인기', '인기순위', '인기검색어',
        # 연예 매체/섹션
        '연예뉴스', '연예면', '스포츠뉴스', '스포츠면',
        # SNS/인스타그램 관련 (연예인들이 자주 언급됨)
        '인스타그램', '인스타', 'sns', '팔로워', '좋아요',
        # 기타 연예 관련
        '화보', '촬영', '제작발표회', '시사회', '시상식', '수상', '시상',
        '커플', '열애', '결혼', '이혼', '출산', '임신', '교제', '열애설',
        '발표', '공개', '공개연애', '열애인정'
    ]
    
    # 법안 관련 키워드 (법안 뉴스에 자주 나타나는 단어)
    legal_terms = [
        '법안', '법률', '법률안', '개정', '입법', '국회', '의원', '정당',
        '정책', '제도', '규제', '법령', '조례', '시행령', '시행규칙',
        '심사', '통과', '가결', '폐기', '처리', '발의', '제안'
    ]
    
    filtered = []
    for news in news_list:
        # 제목, 요약, 본문 확인
        title = str(news.get('title', '')).lower()
        summary = str(news.get('summary', '')).lower()
        body = str(news.get('body', '')).lower()
        
        # 전체 텍스트 결합 (더 강력한 필터링을 위해)
        full_text = f"{title} {summary} {body}"
        
        # 1. 연예/스포츠 관련 키워드가 제목, 요약, 본문 어디에든 있으면 강력하게 제외
        if any(term in full_text for term in entertainment_terms):
            continue
        
        # 2. 제목에 연예/스포츠 키워드가 있으면 무조건 제외 (가장 강력한 필터)
        if any(term in title for term in entertainment_terms):
            continue
        
        # 3. 제목에 법안 관련 키워드가 있으면 우선 포함 (법안 뉴스일 가능성 높음)
        has_legal_term = any(term in title for term in legal_terms)
        
        # 4. 키워드 매칭 확인
        title_has_keyword = any(kw in title for kw in keywords_lower)
        summary_has_keyword = any(kw in summary for kw in keywords_lower)
        body_has_keyword = any(kw in body for kw in keywords_lower)
        
        # 5. 필터링 기준
        # strict_mode에 따라 필터링 기준 조정
        if strict_mode:
            # 엄격 모드: 제목에 키워드가 반드시 포함되어 있어야 함
            # 또는 제목에 법안 관련 키워드가 있으면 포함
            if title_has_keyword or has_legal_term:
                filtered.append(news)
        else:
            # 완화 모드: 제목 또는 요약에 키워드가 있으면 포함
            # 또는 제목에 법안 관련 키워드가 있으면 포함
            if title_has_keyword or summary_has_keyword or has_legal_term:
                filtered.append(news)
    
    return filtered

def search_news_deepsearch(keywords: list, start_date: str, end_date: str, max_results: int = 10) -> list:
    """
    딥서치뉴스 API를 사용하여 뉴스를 검색합니다.
    API 문서: https://news.deepsearch.com/api/
    
    Args:
        keywords: 검색 키워드 리스트
        start_date: 시작 날짜 (YYYY-MM-DD)
        end_date: 종료 날짜 (YYYY-MM-DD)
        max_results: 최대 결과 개수 (최대 100)
    
    Returns:
        뉴스 리스트 (dict 형태)
    """
    if not keywords:
        return []
    
    # 키워드를 하나의 검색어로 결합 (OR 조건)
    # API 문서에 따르면 띄어쓰기 사용 시 큰따옴표나 괄호로 묶어야 함
    # 의미 있는 키워드만 선택 (너무 짧거나 긴 키워드 제외)
    valid_keywords = []
    for kw in keywords:
        kw_clean = kw.strip()
        # 2-20자 사이의 키워드만 사용, 너무 긴 문장은 제외
        if 2 <= len(kw_clean) <= 20 and not (len(kw_clean) > 15 and ' ' in kw_clean and kw_clean.count(' ') > 3):
            valid_keywords.append(kw_clean)
    
    if not valid_keywords:
        return []
    
    # 최대 10개 키워드 사용 (더 많은 키워드로 검색 범위 확대)
    keyword_parts = []
    for kw in valid_keywords[:10]:
        # 띄어쓰기가 있으면 큰따옴표로 묶기
        if ' ' in kw:
            keyword_parts.append(f'"{kw}"')
        else:
            keyword_parts.append(kw)
    
    search_query = " OR ".join(keyword_parts)
    
    # 디버깅: 첫 번째 호출 시에만 검색 쿼리 출력
    if not hasattr(search_news_deepsearch, '_query_logged'):
        print(f"  [DEBUG] 검색 쿼리 (첫 200자): {search_query[:200]}")
        search_news_deepsearch._query_logged = True
    
    try:
        # API 문서에 따른 정확한 파라미터 사용
        # 베이스 URL: https://api-v2.deepsearch.com
        # 엔드포인트: /v1/articles
        url = DEEPSEARCH_API_BASE_URL + DEEPSEARCH_API_ENDPOINT
        
        # 파라미터 구성 (API 문서에 따름)
        params = {
            "keyword": search_query,
            "date_from": start_date,
            "date_to": end_date,
            "page_size": min(max_results, 100),  # 최대 100
            "page": 1,
            "api_key": DEEPSEARCH_API_KEY  # 쿼리 파라미터로 API 키 전달
        }
        
        # GET 요청 (API 문서 예제에 따름)
        response = requests.get(
            url,
            params=params,
            timeout=30
        )
        
        if response.status_code == 200:
            try:
                data = response.json()
                # API 응답 구조: {'detail': ..., 'total_items': ..., 'data': [...]}
                if isinstance(data, dict):
                    # 'data' 키에 뉴스 리스트가 있음 (확인된 구조)
                    if "data" in data and isinstance(data["data"], list):
                        news_list = data["data"]
                        total_items = data.get('total_items', 0)
                        
                        # 디버깅: 매번 응답 정보 출력 (처음 몇 번만)
                        if not hasattr(search_news_deepsearch, '_response_count'):
                            search_news_deepsearch._response_count = 0
                        search_news_deepsearch._response_count += 1
                        
                        if search_news_deepsearch._response_count <= 3:
                            print(f"  [DEBUG] API 응답: total_items={total_items}, data 길이={len(news_list)}")
                            if total_items == 0:
                                print(f"  [DEBUG] total_items가 0입니다. 검색 쿼리: {search_query[:100]}")
                        
                        if len(news_list) > 0:
                            # 뉴스 필터링: 연예 뉴스 제외하고 법안 관련 뉴스만
                            filtered_news = filter_relevant_news(news_list, keywords, strict_mode=True)
                            if len(filtered_news) > 0:
                                return filtered_news[:max_results]
                            
                            # 필터링 후 결과가 없으면 완화 모드로 재시도
                            filtered_relaxed = filter_relevant_news(news_list, keywords, strict_mode=False)
                            if len(filtered_relaxed) > 0:
                                return filtered_relaxed[:max_results]
                            
                            # 여전히 결과가 없으면 연예 뉴스만 제외하고 나머지는 포함 (최소 모드)
                            # 연예 뉴스는 절대 포함하지 않되, 법안 관련 뉴스는 최소한이라도 포함
                            minimal_filtered = []
                            entertainment_terms = [
                                '연예', '가수', '배우', '아이돌', '드라마', '영화', '예능', '엔터테인먼트',
                                '걸그룹', '보이그룹', '연예계', '연예인', 'k-pop', 'kpop',
                                '스포츠', '야구', '축구', '농구', '프로야구', '프로축구', '경기', '선수'
                            ]
                            
                            for news in news_list:
                                title = str(news.get('title', '')).lower()
                                summary = str(news.get('summary', '')).lower()
                                full_text = f"{title} {summary}".lower()
                                
                                # 연예/스포츠 키워드가 제목에 있으면 제외
                                if any(term in title for term in entertainment_terms):
                                    continue
                                
                                # 연예/스포츠 키워드가 요약에만 있고 제목에는 없으면 제외
                                if any(term in summary and term not in title for term in entertainment_terms):
                                    continue
                                
                                # 나머지는 포함 (최소한의 뉴스라도 반환)
                                minimal_filtered.append(news)
                            
                            if len(minimal_filtered) > 0:
                                return minimal_filtered[:max_results]
                            
                            # 정말 아무 뉴스도 없으면 빈 리스트 반환
                            return []
                    # 다른 가능한 키들도 확인 (fallback)
                    news_list = data.get("articles", data.get("items", data.get("results", [])))
                    if news_list and isinstance(news_list, list) and len(news_list) > 0:
                        # 적응형 필터링 적용
                        filtered_strict = filter_relevant_news(news_list, keywords, strict_mode=True)
                        if 5 <= len(filtered_strict) <= 10:
                            return filtered_strict[:max_results]
                        elif len(filtered_strict) < 5:
                            filtered_relaxed = filter_relevant_news(news_list, keywords, strict_mode=False)
                            if len(filtered_relaxed) >= 3:
                                return filtered_relaxed[:max_results]
                            elif len(filtered_strict) > 0:
                                return filtered_strict
                            elif len(filtered_relaxed) > 0:
                                return filtered_relaxed[:max_results]
                            
                            # 최소 모드: 연예 뉴스만 제외
                            minimal_filtered = []
                            entertainment_terms = [
                                '연예', '가수', '배우', '아이돌', '드라마', '영화', '예능', '엔터테인먼트',
                                '걸그룹', '보이그룹', '연예계', '연예인', 'k-pop', 'kpop',
                                '스포츠', '야구', '축구', '농구', '프로야구', '프로축구', '경기', '선수'
                            ]
                            for news in news_list:
                                title = str(news.get('title', '')).lower()
                                if not any(term in title for term in entertainment_terms):
                                    minimal_filtered.append(news)
                            if len(minimal_filtered) > 0:
                                return minimal_filtered[:max_results]
                            return []
                        else:
                            # 10개 이상이면 더 엄격하게
                            double_filtered = [n for n in filtered_strict if sum(1 for kw in keywords_lower if kw in str(n.get('title', '')).lower()) >= 2]
                            if len(double_filtered) >= 5:
                                return double_filtered[:max_results]
                            return filtered_strict[:max_results]
                elif isinstance(data, list) and len(data) > 0:
                    # 뉴스 필터링 적용
                    filtered_news = filter_relevant_news(data, keywords)
                    if len(filtered_news) >= 3:
                        return filtered_news[:max_results]
                    elif len(filtered_news) > 0:
                        return filtered_news
                    
                    # 최소 모드: 연예 뉴스만 제외
                    minimal_filtered = []
                    entertainment_terms = [
                        '연예', '가수', '배우', '아이돌', '드라마', '영화', '예능', '엔터테인먼트',
                        '걸그룹', '보이그룹', '연예계', '연예인', 'k-pop', 'kpop',
                        '스포츠', '야구', '축구', '농구', '프로야구', '프로축구', '경기', '선수'
                    ]
                    for news in data:
                        title = str(news.get('title', '')).lower()
                        if not any(term in title for term in entertainment_terms):
                            minimal_filtered.append(news)
                    if len(minimal_filtered) > 0:
                        return minimal_filtered[:max_results]
                    return []
            except json.JSONDecodeError as e:
                print(f"  JSON 파싱 오류: {e}")
                print(f"  응답 내용 (첫 500자): {response.text[:500]}")
                return []
        else:
            # Authorization Bearer 방식으로 재시도
            try:
                headers = {
                    "Authorization": f"Bearer {DEEPSEARCH_API_KEY}"
                }
                # api_key 파라미터 제거하고 헤더로 전달
                params_without_key = {k: v for k, v in params.items() if k != "api_key"}
                response = requests.get(
                    url,
                    params=params_without_key,
                    headers=headers,
                    timeout=30
                )
                
                if response.status_code == 200:
                    try:
                        data = response.json()
                        if isinstance(data, dict):
                            # 'data' 키에 뉴스 리스트가 있음 (확인된 구조)
                            if "data" in data and isinstance(data["data"], list):
                                news_list = data["data"]
                                if len(news_list) > 0:
                                    # 뉴스 필터링: 제목에 핵심 키워드가 반드시 포함되어 있어야 함
                                    filtered_news = filter_relevant_news(news_list, keywords)
                                    if len(filtered_news) >= 3:
                                        return filtered_news[:max_results]
                                    elif len(filtered_news) > 0:
                                        return filtered_news
                                    
                                    # 최소 모드: 연예 뉴스만 제외
                                    minimal_filtered = []
                                    entertainment_terms = [
                                        '연예', '가수', '배우', '아이돌', '드라마', '영화', '예능', '엔터테인먼트',
                                        '걸그룹', '보이그룹', '연예계', '연예인', 'k-pop', 'kpop',
                                        '스포츠', '야구', '축구', '농구', '프로야구', '프로축구', '경기', '선수'
                                    ]
                                    for news in news_list:
                                        title = str(news.get('title', '')).lower()
                                        if not any(term in title for term in entertainment_terms):
                                            minimal_filtered.append(news)
                                    if len(minimal_filtered) > 0:
                                        return minimal_filtered[:max_results]
                                    return []
                            # 다른 가능한 키들도 확인 (fallback)
                            news_list = data.get("articles", data.get("items", data.get("results", [])))
                            if news_list and isinstance(news_list, list) and len(news_list) > 0:
                                filtered_news = filter_relevant_news(news_list, keywords)
                                if len(filtered_news) >= 3:
                                    return filtered_news[:max_results]
                                elif len(filtered_news) > 0:
                                    return filtered_news
                                
                                # 최소 모드: 연예 뉴스만 제외
                                minimal_filtered = []
                                entertainment_terms = [
                                    '연예', '가수', '배우', '아이돌', '드라마', '영화', '예능', '엔터테인먼트',
                                    '걸그룹', '보이그룹', '연예계', '연예인', 'k-pop', 'kpop',
                                    '스포츠', '야구', '축구', '농구', '프로야구', '프로축구', '경기', '선수'
                                ]
                                for news in news_list:
                                    title = str(news.get('title', '')).lower()
                                    if not any(term in title for term in entertainment_terms):
                                        minimal_filtered.append(news)
                                if len(minimal_filtered) > 0:
                                    return minimal_filtered[:max_results]
                                return []
                        elif isinstance(data, list) and len(data) > 0:
                            filtered_news = filter_relevant_news(data, keywords)
                            if len(filtered_news) >= 3:
                                return filtered_news[:max_results]
                            elif len(filtered_news) > 0:
                                return filtered_news
                            
                            # 최소 모드: 연예 뉴스만 제외
                            minimal_filtered = []
                            entertainment_terms = [
                                '연예', '가수', '배우', '아이돌', '드라마', '영화', '예능', '엔터테인먼트',
                                '걸그룹', '보이그룹', '연예계', '연예인', 'k-pop', 'kpop',
                                '스포츠', '야구', '축구', '농구', '프로야구', '프로축구', '경기', '선수'
                            ]
                            for news in data:
                                title = str(news.get('title', '')).lower()
                                if not any(term in title for term in entertainment_terms):
                                    minimal_filtered.append(news)
                            if len(minimal_filtered) > 0:
                                return minimal_filtered[:max_results]
                            return []
                    except json.JSONDecodeError:
                        pass
            except:
                pass
            
            # 에러 로그 (첫 번째 실패 시에만)
            if not hasattr(search_news_deepsearch, '_error_logged'):
                print(f"  API 호출 실패: 상태 코드 {response.status_code}")
                print(f"  검색 쿼리: {search_query}")
                print(f"  날짜 범위: {start_date} ~ {end_date}")
                if response.status_code == 422:
                    print(f"  422 Validation Error: 파라미터 형식 오류일 수 있습니다.")
                    try:
                        error_data = response.json()
                        print(f"  에러 상세: {error_data}")
                    except:
                        pass
                elif response.status_code == 403:
                    print(f"  403 Forbidden: API 키 인증 문제일 수 있습니다.")
                print(f"  응답 내용 (첫 500자): {response.text[:500]}")
                search_news_deepsearch._error_logged = True
        
        return []
            
    except Exception as e:
        print(f"뉴스 검색 오류: {e}")
        import traceback
        traceback.print_exc()
        return []

# 테스트 (실제 API 호출 전에 주석 처리)
# test_keywords = ["헌법재판소법", "일부개정법률안"]
# test_news = search_news_deepsearch(test_keywords, "2019-01-01", "2019-12-31", 5)
# print(f"테스트 결과: {len(test_news)}개 뉴스")

In [38]:
def get_date_range_for_bill(propose_dt) -> tuple:
    """
    법안 발의일을 기준으로 뉴스 검색 날짜 범위를 계산합니다.
    발의일 전후 6개월 범위로 설정합니다.
    """
    if pd.isna(propose_dt):
        # 기본값: 현재 날짜 기준
        end_date = datetime.now()
        start_date = end_date - timedelta(days=180)
    else:
        if isinstance(propose_dt, str):
            try:
                propose_dt = pd.to_datetime(propose_dt)
            except:
                end_date = datetime.now()
                start_date = end_date - timedelta(days=180)
                return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")
        
        # 발의일 전후 6개월
        start_date = propose_dt - timedelta(days=180)
        end_date = propose_dt + timedelta(days=180)
    
    return start_date.strftime("%Y-%m-%d"), end_date.strftime("%Y-%m-%d")

# 테스트
test_date = pd.to_datetime("2019-10-23")
start, end = get_date_range_for_bill(test_date)
print(f"날짜 범위: {start} ~ {end}")

날짜 범위: 2019-04-26 ~ 2020-04-20


In [39]:
# 데이터베이스에서 필요한 모든 컬럼 가져오기
# news 컬럼이 null이 아니어도 모두 다시 업데이트합니다
with engine.connect() as conn:
    query = text("""
        SELECT bill_id, bill_name, proposer_name, summary, report_md, propose_dt
        FROM public.final_training_data_copy_sample10_md
        WHERE report_md IS NOT NULL 
          AND report_md != ''
          AND report_md != '준비 중'
        ORDER BY propose_dt DESC
    """)
    
    df = pd.read_sql(query, conn)
    
print(f"총 {len(df)}건의 보고서를 처리합니다. (news 컬럼이 이미 있어도 모두 업데이트됩니다)")
print(f"샘플 데이터:")
print(df[['bill_id', 'bill_name', 'proposer_name', 'propose_dt']].head())

총 50건의 보고서를 처리합니다. (news 컬럼이 이미 있어도 모두 업데이트됩니다)
샘플 데이터:
                              bill_id  \
0  PRC_L1Q9D1E0L2O3K1E6P1Q0C1X4V7M2I9   
1  PRC_M1E9S0A5B2U4T1M5L2S0I3I4F3I9Y7   
2  PRC_G1A9B0Y4S0Q8K1T4W5D7N2A8I8T8D6   
3  PRC_X1O8H1U2O0U7U1Z4Z1W9J4U4T2F6F7   
4  PRC_C1U8Y0J7D0U9I1A3I4S2G5P4I7M2R3   

                                       bill_name proposer_name propose_dt  
0  기간제 및 단시간근로자 보호 등에 관한 법률 일부개정법률안(김해영의원 등 11인)           김해영 2019-10-23  
1   인터넷전문은행 설립 및 운영에 관한 특례법 일부개정법률안(김종석의원 등 11인)           김종석 2019-05-24  
2              녹색건축물 조성 지원법 일부개정법률안(안호영의원 등 17인)           안호영 2019-04-08  
3                   종합부동산세법 일부개정법률안(대안)(기획재정위원장)           정성호 2018-12-08  
4                 실내공기질 관리법 일부개정법률안(신보라의원 등 12인)           신보라 2018-07-09  


In [40]:
# 각 레코드에 대해 뉴스 수집 및 저장
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="뉴스 수집 중"):
    bill_id = row['bill_id']
    report_md = row['report_md']
    propose_dt = row['propose_dt']
    
    try:
        # 1. 유사 사례 비교 분석 섹션 추출
        similar_cases_section = extract_similar_cases_section(report_md)
        
        if not similar_cases_section:
            print(f"[{idx+1}/{len(df)}] {bill_id}: 유사 사례 비교 분석 섹션을 찾을 수 없습니다.")
            results.append({
                'bill_id': bill_id,
                'news': json.dumps([], ensure_ascii=False),
                'status': 'no_section'
            })
            continue
        
        # 2. 키워드 추출 (bill_name, proposer_name, summary, similar_cases_section 모두 사용)
        bill_row = {
            'bill_name': row.get('bill_name', ''),
            'proposer_name': row.get('proposer_name', ''),
            'summary': row.get('summary', '')
        }
        keywords = extract_keywords_from_bill_data(bill_row, similar_cases_section)
        
        if not keywords:
            print(f"[{idx+1}/{len(df)}] {bill_id}: 키워드를 추출할 수 없습니다.")
            results.append({
                'bill_id': bill_id,
                'news': json.dumps([], ensure_ascii=False),
                'status': 'no_keywords'
            })
            continue
        
        print(f"[{idx+1}/{len(df)}] {bill_id}: 키워드 {len(keywords)}개 추출")
        print(f"  주요 키워드: {keywords[:5]}")
        
        # 3. 날짜 범위 계산
        start_date, end_date = get_date_range_for_bill(propose_dt)
        
        # 4. 뉴스 검색 (더 정확한 검색을 위해 키워드 수 제한)
        # 최대 30개 뉴스를 가져온 후 적응형 필터링하여 관련성 높은 뉴스만 선택
        # 적응형 필터링으로 5-10개 정도로 일관되게 반환
        news_list = search_news_deepsearch(keywords, start_date, end_date, max_results=30)
        
        # 디버깅: 첫 번째 검색 시 상세 정보
        if idx == 0 and not news_list:
            print(f"  [DEBUG] 첫 번째 검색 실패 - 키워드: {keywords[:5]}")
            print(f"  [DEBUG] 날짜 범위: {start_date} ~ {end_date}")
        
        if not news_list:
            print(f"[{idx+1}/{len(df)}] {bill_id}: 뉴스를 찾을 수 없습니다.")
            results.append({
                'bill_id': bill_id,
                'news': json.dumps([], ensure_ascii=False),
                'status': 'no_news'
            })
        else:
            print(f"[{idx+1}/{len(df)}] {bill_id}: {len(news_list)}개 뉴스 발견")
            results.append({
                'bill_id': bill_id,
                'news': json.dumps(news_list, ensure_ascii=False),
                'status': 'success'
            })
        
        # API 호출 제한을 위한 딜레이
        time.sleep(0.5)
        
    except Exception as e:
        print(f"[{idx+1}/{len(df)}] {bill_id}: 오류 발생 - {e}")
        results.append({
            'bill_id': bill_id,
            'news': json.dumps([], ensure_ascii=False),
            'status': f'error: {str(e)}'
        })

print(f"\n처리 완료: {len(results)}건")

뉴스 수집 중:   0%|          | 0/50 [00:00<?, ?it/s]

[1/50] PRC_L1Q9D1E0L2O3K1E6P1Q0C1X4V7M2I9: 키워드 3개 추출
  주요 키워드: ['보호', '기간제', '단시간근로자']
  [DEBUG] 검색 쿼리 (첫 200자): 보호 OR 기간제 OR 단시간근로자
  [DEBUG] API 응답: total_items=317390, data 길이=27
[1/50] PRC_L1Q9D1E0L2O3K1E6P1Q0C1X4V7M2I9: 1개 뉴스 발견


뉴스 수집 중:   2%|▏         | 1/50 [00:01<01:27,  1.78s/it]

[2/50] PRC_M1E9S0A5B2U4T1M5L2S0I3I4F3I9Y7: 키워드 4개 추출
  주요 키워드: ['인터넷전문은행', '운영', '특례법', '설립']
  [DEBUG] API 응답: total_items=844282, data 길이=30
[2/50] PRC_M1E9S0A5B2U4T1M5L2S0I3I4F3I9Y7: 28개 뉴스 발견


뉴스 수집 중:   4%|▍         | 2/50 [00:03<01:25,  1.79s/it]

...

[49/50] PRC_Y0K8K0G7A2P8Y1I7L5L3M1E9W9U2N8: 키워드 3개 추출
  주요 키워드: ['교육관련기관', '정보공개', '특례법']
[49/50] PRC_Y0K8K0G7A2P8Y1I7L5L3M1E9W9U2N8: 2개 뉴스 발견


뉴스 수집 중:  98%|█████████▊| 49/50 [01:20<00:01,  1.62s/it]

[50/50] PRC_R0M8M0K7T1V1M1H4X1H1M0F2N9W5Q1: 키워드 3개 추출
  주요 키워드: ['염관리법', '중소기업협동조합법', '염업조합법']
[50/50] PRC_R0M8M0K7T1V1M1H4X1H1M0F2N9W5Q1: 4개 뉴스 발견


뉴스 수집 중: 100%|██████████| 50/50 [01:22<00:00,  1.64s/it]


처리 완료: 50건


In [41]:
# 결과를 데이터프레임으로 변환
results_df = pd.DataFrame(results)
print("결과 통계:")
print(results_df['status'].value_counts())
print(f"\n성공: {len(results_df[results_df['status'] == 'success'])}건")
print(f"실패: {len(results_df[results_df['status'] != 'success'])}건")

결과 통계:
status
success    50
Name: count, dtype: int64

성공: 50건
실패: 0건


In [42]:
# 데이터베이스에 news 컬럼 업데이트
# news 컬럼이 null이 아니어도 모두 덮어씁니다
with engine.connect() as conn:
    updated_count = 0
    for _, row in results_df.iterrows():
        try:
            update_query = text("""
                UPDATE public.final_training_data_copy_sample10_md
                SET news = :news
                WHERE bill_id = :bill_id
            """)
            conn.execute(update_query, {
                'news': row['news'],
                'bill_id': row['bill_id']
            })
            updated_count += 1
        except Exception as e:
            print(f"업데이트 실패 ({row['bill_id']}): {e}")
    
    conn.commit()
    print(f"\n데이터베이스 업데이트 완료: {updated_count}건 (기존 news 값이 있어도 모두 덮어썼습니다)")


데이터베이스 업데이트 완료: 50건 (기존 news 값이 있어도 모두 덮어썼습니다)


In [43]:
# 샘플 결과 확인
sample_success = results_df[results_df['status'] == 'success'].head(1)
if len(sample_success) > 0:
    sample_news = json.loads(sample_success.iloc[0]['news'])
    print(f"샘플 뉴스 ({sample_success.iloc[0]['bill_id']}):")
    print(f"뉴스 개수: {len(sample_news)}")
    if sample_news:
        print(f"첫 번째 뉴스:")
        print(json.dumps(sample_news[0], indent=2, ensure_ascii=False))

샘플 뉴스 (PRC_L1Q9D1E0L2O3K1E6P1Q0C1X4V7M2I9):
뉴스 개수: 1
첫 번째 뉴스:
{
  "id": "903861668772188452",
  "sections": [
    "society"
  ],
  "title": "‘총선민심 확인‘ 文 대통령, “국민 믿고 담대하게“ 코로나 잡고 경제도 살린다",
  "publisher": "세계일보",
  "author": "현화영",
  "summary": "문 대통령은 “방역에서부터 세계의 희망이 되는 나라가 되겠다”면서 코로나19 위기 극복이 일 순위임을 강조했다.\n\n문 대통령은 “경제에서도 전 세계에 위기 극복의 저력을 보여주는 나라가 되겠다”면서 방역의 성과를 경제로 연결 짓고, 선제적이며 과감한 정책으로 경제 회복의 시간을 앞당기겠다”고 했다.\n\n문 대통령은 “K-방역에 이어 K-경제까지 위기 극복의 세계적 표준이 되겠다”고 했다.",
  "highlight": null,
  "score": null,
  "image_url": "https://ddi-cdn.deepsearch.com/news/society/2020/04/20/903861668772188452/000-87b53725ad5d7e9e9a584f69180f80fbcec34e0c.jpg",
  "thumbnail_url": null,
  "content_url": "http://www.segye.com/content/html/2020/04/20/20200420518669.html",
  "esg": null,
  "companies": [],
  "entities": [],
  "named_entities": [
    {
      "type": "company",
      "exchange": null,
      "market": null,
      "symbol": "NICE:496590",
      "name": "더불어민주당",
      "ceo": [
        {
          "n